In [ ]:
import os, gc
from pathlib import Path
import numpy as np
import xarray as xr
import rioxarray
import cdsapi
import rasterio

# CPU stability
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

# Global parameters
YEARS  = list(range(2010, 2025))
MONTHS = list(range(1, 13))

ROOT    = Path(".")
OUT_DIR = ROOT / "../data/hazards/cold"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Country bounding boxes (ERA5 format: [N, W, S, E])
COUNTRIES = {
   #  "TJK": {"area": [41.1, 67.3, 36.5, 75.2]},
    "TKM": {"area": [42.8, 52.2, 35.0, 66.8]},
   #  "KGZ": {"area": [43.3, 69.0, 39.0, 80.0]},
    # "KAZ": {"area": [55.5, 46.0, 40.0, 87.5]},
   #  "UZB": {"area": [46.0, 55.0, 37.0, 74.0]},
}

# Helpers
def cds_client():
    return cdsapi.Client()

def download_hourly_month(c, year, month, out_nc, area_bbox):
    if out_nc.exists():
        print(f"[skip] {out_nc.name}")
        return
    print(f"[download] ERA5 hourly t2m {year}-{month:02d}")
    c.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable": "2m_temperature",
            "year": str(year),
            "month": f"{month:02d}",
            "day": [f"{d:02d}" for d in range(1, 32)],
            "time": [f"{h:02d}:00" for h in range(24)],
            "area": area_bbox,
            "format": "netcdf",
        },
        str(out_nc),
    )

def detect_time_dim(da):
    if "time" in da.dims:
        return "time"
    if "valid_time" in da.dims:
        return "valid_time"
    raise ValueError("No time dimension found.")

def ensure_spatial(da):
    da = da.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude", inplace=False)
    da = da.rio.write_crs("EPSG:4326", inplace=False)
    return da

def quick_stats(path):
    with rasterio.open(path) as src:
        arr = src.read(1)
        nod = src.nodata
    arr = arr.astype("float32")
    if nod is not None:
        vals = arr[arr != nod]
    else:
        vals = arr[np.isfinite(arr)]
    return (
        float(np.nanmin(vals)),
        float(np.nanmean(vals)),
        float(np.nanmax(vals)),
    )

# Main loop
c = cds_client()

for COUNTRY, cfg in COUNTRIES.items():
    area_bbox = cfg["area"]
    print("\n" + "#" * 80)
    print(f"### COUNTRY: {COUNTRY} | AREA = {area_bbox}")
    print("#" * 80)

    TMP_DIR = ROOT / f"./_tmp_era5_{COUNTRY.lower()}_cold_allmonths"
    TMP_DIR.mkdir(parents=True, exist_ok=True)

    OUT_TNN = OUT_DIR / f"TNn_mean_2010_2024_{COUNTRY}.tif"
    OUT_FD  = OUT_DIR / f"FD_mean_2010_2024_{COUNTRY}.tif"

    tnn_years = []
    fd_years  = []

    for y in YEARS:
        print(f"\n=== {COUNTRY} — YEAR {y} ===")
        tnn_year = None
        fd_year  = None

        for m in MONTHS:
            hourly_nc = TMP_DIR / f"era5_t2m_hourly_{COUNTRY.lower()}_{y}_{m:02d}.nc"
            download_hourly_month(c, y, m, hourly_nc, area_bbox)

            print(f"[process] {hourly_nc.name}")
            ds = xr.open_dataset(hourly_nc, chunks={"time": 240})
            da_hour = (ds["t2m"] - 273.15).rename("t2m")

            tdim = detect_time_dim(da_hour)

            daily_tmin = da_hour.resample({tdim: "1D"}).min(skipna=True)
            fd_month   = (daily_tmin < 0).sum(dim=tdim).astype("float32")
            tnn_month  = daily_tmin.min(dim=tdim)

            tnn_year = tnn_month if tnn_year is None else xr.ufuncs.minimum(tnn_year, tnn_month)
            fd_year  = fd_month if fd_year is None else (fd_year + fd_month)

            ds.close()
            del ds, da_hour, daily_tmin, fd_month, tnn_month
            gc.collect()

        tnn_year = ensure_spatial(tnn_year).astype("float32")
        fd_year  = ensure_spatial(fd_year).astype("float32")

        tnn_years.append(tnn_year)
        fd_years.append(fd_year)

        print(
            f"[{COUNTRY} {y}] "
            f"TNn range {float(tnn_year.min()):.1f}..{float(tnn_year.max()):.1f} °C | "
            f"FD range {float(fd_year.min()):.0f}..{float(fd_year.max()):.0f}"
        )

    print(f"\n[aggregate] Multi-annual means for {COUNTRY}")

    tnn_stack = xr.concat(tnn_years, dim="year")
    fd_stack  = xr.concat(fd_years,  dim="year")

    tnn_mean = tnn_stack.mean(dim="year", skipna=True)
    fd_mean  = fd_stack.mean(dim="year", skipna=True)

    tnn_mean.rio.to_raster(OUT_TNN)
    fd_mean.rio.to_raster(OUT_FD)

    print("\n✔ Written:")
    print(" -", OUT_TNN)
    print(" -", OUT_FD)
    print("TNn stats:", quick_stats(OUT_TNN))
    print("FD stats :", quick_stats(OUT_FD))



################################################################################
### COUNTRY: TKM | AREA = [42.8, 52.2, 35.0, 66.8]
################################################################################

=== TKM — YEAR 2010 ===
[skip] era5_t2m_hourly_tkm_2010_01.nc
[process] era5_t2m_hourly_tkm_2010_01.nc
[skip] era5_t2m_hourly_tkm_2010_02.nc
[process] era5_t2m_hourly_tkm_2010_02.nc
[skip] era5_t2m_hourly_tkm_2010_03.nc
[process] era5_t2m_hourly_tkm_2010_03.nc
[skip] era5_t2m_hourly_tkm_2010_04.nc
[process] era5_t2m_hourly_tkm_2010_04.nc
[skip] era5_t2m_hourly_tkm_2010_05.nc
[process] era5_t2m_hourly_tkm_2010_05.nc
[skip] era5_t2m_hourly_tkm_2010_06.nc
[process] era5_t2m_hourly_tkm_2010_06.nc
[skip] era5_t2m_hourly_tkm_2010_07.nc
[process] era5_t2m_hourly_tkm_2010_07.nc
[skip] era5_t2m_hourly_tkm_2010_08.nc
[process] era5_t2m_hourly_tkm_2010_08.nc
[skip] era5_t2m_hourly_tkm_2010_09.nc
[process] era5_t2m_hourly_tkm_2010_09.nc
[skip] era5_t2m_hourly_tkm_2010_10.nc
[process] er

2025-12-03 16:50:06,496 INFO Request ID is 74a2fcfd-5c35-4d60-b5f1-bb11e0210d22
2025-12-03 16:50:06,579 INFO status has been updated to accepted
2025-12-03 16:50:28,005 INFO status has been updated to successful


60f55e3bff4f5fae99f81341719e10f8.nc:   0%|          | 0.00/2.52M [00:00<?, ?B/s]

[process] era5_t2m_hourly_tkm_2013_04.nc
[download] ERA5 hourly t2m 2013-05


2025-12-03 16:50:29,663 INFO Request ID is 0b4843d9-9db2-400f-8b00-b6725e1ab53e
2025-12-03 16:50:29,758 INFO status has been updated to accepted
2025-12-03 16:50:43,500 INFO status has been updated to running
2025-12-03 16:52:24,314 INFO status has been updated to successful


...

[process] era5_t2m_hourly_tkm_2013_10.nc
[download] ERA5 hourly t2m 2013-11


2025-12-03 17:00:13,105 INFO Request ID is 5ee8ce34-7260-4b08-a2a2-7da922f7c019
2025-12-03 17:00:13,198 INFO status has been updated to accepted
2025-12-03 17:10:40,279 INFO status has been updated to successful


edda69440745f3ea927d4ff9bd07c493.nc:   0%|          | 0.00/2.52M [00:00<?, ?B/s]

[process] era5_t2m_hourly_tkm_2013_11.nc
[download] ERA5 hourly t2m 2013-12


2025-12-03 17:10:41,888 INFO Request ID is 0f0ba203-500c-45a5-a3bf-9a9fd4f2fe8c
2025-12-03 17:10:41,998 INFO status has been updated to accepted


In [1]:
# -------------------
#  In case of the kernel is full
# -------------------
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"